In [ ]:
# Quick sanity check that the notebook kernel is working
3 * 5


In [ ]:
%pip install requests pandas tqdm


In [3]:
# Set COMTRADE_API_KEY in your shell or notebook environment for the full data endpoint.
# Without it, the script uses the public preview endpoint.
%run extract_trade_data.py


Fetching UN Comtrade data: 100%|██████████| 56/56 [01:41<00:00,  1.81s/it]


Dataset Shape:
(168, 11)

Sample Data:
   year reporter  reporter_code partner  partner_code    flow flow_code  \
0  2018    China            156   World             0  Import         M   
1  2018    China            156   World             0  Import         M   
2  2018    China            156   World             0  Import         M   
3  2018    China            156   World             0  Export         X   
4  2018    China            156   World             0  Export         X   

  hs_code                                          commodity  trade_value_usd  \
0  854231  Electronic integrated circuits; processors and...     1.274485e+11   
1  854232           Electronic integrated circuits; memories     1.230867e+11   
2  854239  Electronic integrated circuits; n.e.c. in head...     5.177658e+10   
3  854231  Electronic integrated circuits; processors and...     2.966158e+10   
4  854232           Electronic integrated circuits; memories     4.409389e+10   

   net_weight_kg  
0  

In [4]:
import requests
import pandas as pd
import time
from itertools import product

# ==========================================
# CONFIGURATION
# ==========================================

API_KEY = "c540b6fd190e428bbc5f0b0c862d824b"

BASE_URL = "https://comtradeapi.worldbank.org/data/v1/get/C/A/HS"

YEARS = list(range(2018, 2025))

COUNTRIES = {
    "China": "156",
    "USA": "842",
    "Japan": "392",
    "Korea": "410",
    "Taiwan": "158",
    "Netherlands": "528",
    "Singapore": "702",
    "Germany": "276",
    "India": "356"
}

HS_CODES = [
    "854231",
    "854232",
    "854239"
]

FLOWS = {
    "Imports": "M",
    "Exports": "X"
}

OUTPUT_FILE = "v1_semiconductor_supply_chain.csv"

# ==========================================
# DATA COLLECTION
# ==========================================

all_rows = []

country_pairs = list(product(COUNTRIES.items(), COUNTRIES.items()))

for (reporter_name, reporter_code), (partner_name, partner_code) in country_pairs:

    # Skip same-country trade
    if reporter_code == partner_code:
        continue

    for year in YEARS:

        for flow_name, flow_code in FLOWS.items():

            print(f"Fetching: {reporter_name} -> {partner_name} | {year} | {flow_name}")

            params = {
                "reporter": reporter_code,
                "partner": partner_code,
                "flow": flow_code,
                "commodityCodes": ",".join(HS_CODES),
                "period": year,
                "format": "JSON",
                "maxRecords": 50000
            }

            headers = {
                "Ocp-Apim-Subscription-Key": API_KEY
            }

            try:

                response = requests.get(
                    BASE_URL,
                    params=params,
                    headers=headers
                )

                if response.status_code != 200:
                    print(f"Failed: {response.status_code}")
                    continue

                data = response.json()

                if "data" not in data:
                    print("No data")
                    continue

                records = data["data"]

                for row in records:

                    extracted = {
                        "year": row.get("period"),
                        "reporter": row.get("reporterDesc"),
                        "reporter_code": row.get("reporterCode"),
                        "partner": row.get("partnerDesc"),
                        "partner_code": row.get("partnerCode"),
                        "flow": row.get("flowDesc"),
                        "flow_code": row.get("flowCode"),
                        "hs_code": row.get("cmdCode"),
                        "commodity": row.get("cmdDesc"),
                        "trade_value_usd": row.get("primaryValue"),
                        "net_weight_kg": row.get("netWgt")
                    }

                    all_rows.append(extracted)

                # Avoid API rate limits
                time.sleep(1)

            except Exception as e:
                print(f"Error: {e}")

# ==========================================
# SAVE DATASET
# ==========================================

df = pd.DataFrame(all_rows)

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nSample:")
print(df.head())

df.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved to {OUTPUT_FILE}")

Fetching: China -> USA | 2018 | Imports
Error: HTTPSConnectionPool(host='comtradeapi.worldbank.org', port=443): Max retries exceeded with url: /data/v1/get/C/A/HS?reporter=156&partner=842&flow=M&commodityCodes=854231%2C854232%2C854239&period=2018&format=JSON&maxRecords=50000 (Caused by NameResolutionError("HTTPSConnection(host='comtradeapi.worldbank.org', port=443): Failed to resolve 'comtradeapi.worldbank.org' ([Errno 8] nodename nor servname provided, or not known)"))
Fetching: China -> USA | 2018 | Exports
Error: HTTPSConnectionPool(host='comtradeapi.worldbank.org', port=443): Max retries exceeded with url: /data/v1/get/C/A/HS?reporter=156&partner=842&flow=X&commodityCodes=854231%2C854232%2C854239&period=2018&format=JSON&maxRecords=50000 (Caused by NameResolutionError("HTTPSConnection(host='comtradeapi.worldbank.org', port=443): Failed to resolve 'comtradeapi.worldbank.org' ([Errno 8] nodename nor servname provided, or not known)"))
Fetching: China -> USA | 2019 | Imports
Error: HTT